In [ ]:
# ======================================================
# 1. Installer les libs
# ======================================================
!pip install transformers datasets peft accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 12.1 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

login(token="")

In [ ]:
# ======================================================
# 1. Imports
# ======================================================
import torch
import pandas as pd
import json
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

# ======================================================
# 2. Configs
# ======================================================
MODEL_NAME = "google/flan-t5-xl"
DATA_FILE = "/content/seqFR.jsonl"
OUTPUT_DIR = "./flan-t5-xl-sequence"
MAX_LENGTH = 1024
EPOCHS = 5
BATCH_SIZE = 1
GRAD_ACCUM = 8

# ======================================================
# 3. Charger modèle et tokenizer
# ======================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# ======================================================
# 4. Charger dataset
# ======================================================
df = pd.read_json(DATA_FILE, lines=True)
dataset = Dataset.from_pandas(df)

# ======================================================
# 5. Fonction récursive pour flatten interactions
# ======================================================
def flatten_interactions_recursive(node):
    messages = []
    events = []

    if node is None:
        return messages, events

    if isinstance(node, list):
        for item in node:
            m, e = flatten_interactions_recursive(item)
            messages.extend(m)
            events.extend(e)
        return messages, events

    if not isinstance(node, dict):
        return messages, events

    if "events" in node or "else" in node:
        event_entry = {
            "condition": node.get("condition", "N/A"),
            "text": node.get("text", "N/A"),
            "name": node.get("name", "N/A"),
        }
        event_messages, event_events = [], []
        for key in ("messages",):
            if key in node:
                m, e = flatten_interactions_recursive(node[key])
                event_messages.extend(m)
                event_events.extend(e)
        if event_messages:
            event_entry["messages"] = event_messages
        if event_events:
            event_entry["events"] = event_events
        events.append(event_entry)
        return messages, events

    if "from" in node and "to" in node:
        msg_entry = {
            "from": node.get("from", "N/A"),
            "to": node.get("to", "N/A"),
            "text": node.get("text", "N/A"),
            "name": node.get("name", "N/A"),
            "type": node.get("type", "N/A")
        }
        messages.append(msg_entry)
        for key in ("messages", "events", "interactions", "else"):
            if key in node:
                m, e = flatten_interactions_recursive(node[key])
                messages.extend(m)
                events.extend(e)
        return messages, events

    for key in ("messages", "events", "interactions", "else"):
        if key in node:
            m, e = flatten_interactions_recursive(node[key])
            messages.extend(m)
            events.extend(e)

    return messages, events

# ======================================================
# 6. Formatter dataset pour fine-tuning
# ======================================================
def format_sequence_data_jsonl(ex):
    output_actors = [
        {"text": a.get("text", "N/A"), "name": a.get("name", "N/A"), "alias": a.get("alias", "<no alias>")}
        for a in ex.get("output", {}).get("Actors", [])
    ]

    output_participants = [
        {"text": p.get("text", "N/A"), "name": p.get("name", "N/A"), "alias": p.get("alias", "<no alias>")}
        for p in ex.get("output", {}).get("Participants ", [])
    ]

    root_interactions = ex.get("output", {}).get("interactions") or []
    root_events = ex.get("output", {}).get("events") or []
    all_nodes = root_interactions + root_events

    messages, events = flatten_interactions_recursive(all_nodes)

    json_output = json.dumps({
        "actors": output_actors,
        "participants": output_participants,
        "interactions": {
            "messages": messages,
            "events": events
        }
    }, ensure_ascii=False)

    # Pour T5, on définit input = contexte et target = texte JSON
    return {
        "input_text": ex.get("input", "Describe the UML sequence diagram:"),
        "target_text": json_output
    }

# ======================================================
# 7. Appliquer formatage
# ======================================================
formatted_dataset = dataset.map(format_sequence_data_jsonl)

# ======================================================
# 8. Tokenization
# ======================================================
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    labels = tokenizer(
        examples["target_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset.column_names
)

# ======================================================
# 9. Préparer LoRA
# ======================================================
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=128,
    lora_alpha=32,
    target_modules=["q", "v"],  # pour T5
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# ======================================================
# 10. TrainingArguments
# ======================================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
)

# ======================================================
# 11. Trainer avec DataCollator
# ======================================================
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=peft_model,
)

trainer = Trainer(
    model=peft_model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator,
)

# ======================================================
# 12. Lancer fine-tuning
# ======================================================
trainer.train()

# ======================================================
# 13. Sauvegarde
# ======================================================
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modèle et tokenizer sauvegardés dans {OUTPUT_DIR}")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

trainable params: 51,118,080 || all params: 2,665,459,968 || trainable%: 1.9178


It is strongly recommended to train Gemma2 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.148300
20,0.772500
30,0.694800
40,0.647200
50,0.624100
60,0.617400
70,0.613500
80,0.558800
90,0.555500
100,0.530100


Modèle et tokenizer sauvegardés dans ./gemma-2-2b-it-sequence


In [ ]:
# ======================================================
# 1. Imports
# ======================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ======================================================
# 2. Config
# ======================================================
MODEL_DIR = "/content/drive/MyDrive/flan-t5-xl-sequence"  # chemin vers ton modèle
MAX_LENGTH = 1600

# ======================================================
# 3. Charger le modèle et tokenizer sur GPU
# ======================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16 if device=="cuda" else torch.float32,  # float16 sur GPU
    device_map="auto",  # répartit sur GPU automatiquement
    trust_remote_code=True,
    local_files_only=True
)
model.to(device)
model.eval()

# ======================================================
# 4. Fonction d'inférence
# ======================================================
def predict(input_text):
    prompt = f"Convert this text into a JSON UML Sequence Diagram with actors, participants, and interactions (type: message, event, loop, alt, opt, else), automatically detecting loops, conditional branches, and optional actions.\nInput: {input_text}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_LENGTH,
            temperature=0.0,  # déterministe
            do_sample=False
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return decoded.split("Output:")[-1].strip() if "Output:" in decoded else decoded.strip()

# ======================================================
# 5. Tester avec input
# ======================================================
while True:
    user_input = input("\nEntrez votre texte (ou 'exit' pour quitter) : ")
    if user_input.lower() == "exit":
        break

    prediction = predict(user_input)
    print("\n=== Predicted Structured Output ===")
    print(prediction)


Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Entrez votre texte (ou 'exit' pour quitter) : use send data to the systme with run LOOP 


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Predicted Structured Output ===
JSON representation of the sequence diagram

```text
Input: use send data to the systme with run LOOP 
```

```json
{
  "actors": [
    {
      "name": "user",
      "text": "User"
    },
    {
      "name": "system",
      "text": "System"
    }
  ],
  "participants": [],
  "interactions": [
    {
      "from": "user",
      "to": "system",
      "text": "send data",
      "name": "send data",
      "type": "sync"
    },
    {
      "from": "user",
      "to": "system",
      "text": "run LOOP",
      "name": "run LOOP",
      "type": "sync"
    },
    {
      "from": "system",
      "to": "system",
      "text": "process data",
      "name": "process data",
      "type": "sync"
    },
    {
      "from": "system",
      "to": "system",
      "text": "loop",
      "name": "loop",
      "type": "sync"
    }
  ]
}
```

Entrez votre texte (ou 'exit' pour quitter) : event condition LOOP 

=== Predicted Structured Output ===
sequence diagram

```text
ev

In [ ]:
# ======================================================
# 1. Imports
# ======================================================
import torch
import re
import json
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer

# ======================================================
# 2. Config
# ======================================================
MODEL_DIR = "/content/drive/MyDrive/Modeles/gemma-2-2b-it-sequence-fine-tuned"
MAX_LENGTH = 1600
REF_FILE = "/content/seppeval.jsonl"      # fichier référence
PRED_FILE = "/content/pred.jsonl"    # fichier résultats

# ======================================================
# 3. Charger modèle et tokenizer
# ======================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True
)
model.to(device)
model.eval()

# ======================================================
# 4. Fonction de prédiction
# ======================================================
def predict(input_text):
    prompt = (
        "Convert the following text into a structured JSON representing the UML sequence diagram, "
        "including actors, participants, and interactions (messages and events). "
        "Always output valid JSON only.\n\n"
        f"Input: {input_text}\nOutput:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_LENGTH,
            temperature=0.0,
            do_sample=False
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    match = re.search(r"\{[\s\S]*\}", decoded)
    if match:
        json_only = match.group(0)
        try:
            return json.loads(json_only)
        except json.JSONDecodeError:
            return {}
    return {}

# ======================================================
# 5. Extraction par catégories
# ======================================================
def extract_by_category(obj):
    actors, participants, messages, events = [], [], [], []

    # Actors
    for a in obj.get("Actors", []):
        if "name" in a:
            actors.append(a["name"])

    # Participants
    for p in obj.get("Participants", []):
        if "name" in p:
            participants.append(p["name"])

    # Messages & Events
    for inter in obj.get("interactions", []):
        # Cas direct : interaction est un message
        if "from" in inter and "to" in inter and "name" in inter:
            messages.append(inter["name"])

        # Messages imbriqués
        for m in inter.get("messages", []):
            if "name" in m:
                messages.append(m["name"])
            if "events" in m:
                for ev in m["events"]:
                    if "name" in ev:
                        events.append(ev["name"])
                    for m2 in ev.get("messages", []):
                        if "name" in m2:
                            messages.append(m2["name"])
                    if "else" in ev:
                        if "name" in ev["else"]:
                            events.append(ev["else"]["name"])
                        for m3 in ev["else"].get("messages", []):
                            if "name" in m3:
                                messages.append(m3["name"])

        # Events directs dans interactions
        if "events" in inter:
            for ev in inter["events"]:
                if "name" in ev:
                    events.append(ev["name"])
                for m2 in ev.get("messages", []):
                    if "name" in m2:
                        messages.append(m2["name"])
                if "else" in ev:
                    if "name" in ev["else"]:
                        events.append(ev["else"]["name"])
                    for m3 in ev["else"].get("messages", []):
                        if "name" in m3:
                            messages.append(m3["name"])

    return {
        "actors": actors,
        "participants": participants,
        "messages": messages,
        "events": events,
    }

# ======================================================
# 6. Jaccard et ROUGE
# ======================================================
def jaccard_similarity(list1, list2):
    set1, set2 = set(list1), set(list2)
    if not set1 and not set2:
        return 1.0
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0.0

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l_score(list1, list2):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0
    scores = []
    for ref in list1:
        best = 0.0
        for pred in list2:
            s = scorer.score(ref, pred)["rougeL"].fmeasure
            if s > best:
                best = s
        scores.append(best)
    return sum(scores) / len(scores)

# ======================================================
# 7. Évaluation
# ======================================================
def evaluate_model(ref_file, pred_file):
    results = []
    global_scores = {
        "actors_jaccard": [], "actors_rouge": [],
        "participants_jaccard": [], "participants_rouge": [],
        "messages_jaccard": [], "messages_rouge": [],
        "events_jaccard": [], "events_rouge": []
    }

    with open(ref_file, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Evaluation"):
            ex = json.loads(line.strip())
            input_text = ex.get("input", "")
            ref_output = ex.get("output", {})

            pred_output = predict(input_text)

            ref_cats = extract_by_category(ref_output)
            pred_cats = extract_by_category(pred_output)

            scores = {}
            for cat in ["actors", "participants", "messages", "events"]:
                j = jaccard_similarity(ref_cats[cat], pred_cats[cat])
                r = rouge_l_score(ref_cats[cat], pred_cats[cat])
                scores[f"{cat}_jaccard"] = j
                scores[f"{cat}_rouge"] = r
                global_scores[f"{cat}_jaccard"].append(j)
                global_scores[f"{cat}_rouge"].append(r)

            results.append({
                "input": input_text,
                "output_ref": ref_output,
                "output_pred": pred_output,
                "scores": scores
            })

    # Sauvegarde
    with open(pred_file, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"\n✅ Résultats sauvegardés dans {pred_file}")

    # Scores par catégorie
    for cat in ["actors", "participants", "messages", "events"]:
        j_avg = sum(global_scores[f"{cat}_jaccard"]) / len(global_scores[f"{cat}_jaccard"]) if global_scores[f"{cat}_jaccard"] else 0
        r_avg = sum(global_scores[f"{cat}_rouge"]) / len(global_scores[f"{cat}_rouge"]) if global_scores[f"{cat}_rouge"] else 0
        print(f"📊 {cat.capitalize()} - Jaccard: {j_avg:.3f}, ROUGE-L: {r_avg:.3f}")

    # Score général
    all_jaccard = []
    all_rouge = []
    for cat in ["actors", "participants", "messages", "events"]:
        all_jaccard.extend(global_scores[f"{cat}_jaccard"])
        all_rouge.extend(global_scores[f"{cat}_rouge"])
    general_jaccard = sum(all_jaccard) / len(all_jaccard) if all_jaccard else 0
    general_rouge = sum(all_rouge) / len(all_rouge) if all_rouge else 0

    print(f"\n🌟 Score général - Jaccard: {general_jaccard:.3f}, ROUGE-L: {general_rouge:.3f}")

# ======================================================
# 8. Exécution
# ======================================================
if __name__ == "__main__":
    evaluate_model(REF_FILE, PRED_FILE)


Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluation: 22it [17:27, 47.61s/it]


✅ Résultats sauvegardés dans /content/pred.jsonl
📊 Actors - Jaccard: 0.182, ROUGE-L: 0.182
📊 Participants - Jaccard: 1.000, ROUGE-L: 1.000
📊 Messages - Jaccard: 0.039, ROUGE-L: 0.227
📊 Events - Jaccard: 0.636, ROUGE-L: 0.636

🌟 Score général - Jaccard: 0.464, ROUGE-L: 0.511
